### **Student Information**
GitHub ID: brendatan99

Kaggle name: brendatt

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

***

# **Project Report**

**Syntax:** `#` creates the largest heading (H1).

---
**Syntax:** `---` creates a horizontal rule (a separator line).

## 1. Model Development (10 pts Required)

**Syntax:** `##` creates a secondary heading (H2).

**Describe briefly each section, you can add graphs/charts to support your explanations.**

### 1.1 Preprocessing Steps

**Syntax:** `###` creates a tertiary heading (H3).

[Content for Preprocessing]

**Example Syntax for Content:**
*   **Bold text:** `**text**`
*   *Italic text*: `*text*`
*   Bullet point list:
    * Item 1
    * Item 2

Markdown Syntax to Add Image: `![Description of the Image](./your_local_folder/name_of_the_image.png)`

![Example Markdown Syntax to Add Image](./pics/example_md_img.png)

### 1.2 Feature Engineering Steps

[Content for Feature Engineering]

### 1.3 Explanation of Your Model

[Content for Model Explanation]

### 1.4 AI Prompts, what and why you decide to use that technique that suggest by AI

[Content for AI Prompts and your reason]

---

**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

In [1]:
### Add the code related to the preprocessing steps in cells inside this section

# T0 step 1 - load the three competition files WITHOUT assuming any column name.
# The Kaggle Data page calls the label column "emotion" but the real header is "label",
# so every column list is printed and the printed names are what the rest of the code uses.
import json, re, time, hashlib, warnings
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42          # every randomised step reuses this one seed
N_SPLITS = 5               # one fixed 5-fold split is shared by every experiment
DATA_DIR = Path("data/kaggle")                 # competition files (kept out of git)
OUT_DIR = Path("../_workbench/kaggle")         # experiment records, never committed
for sub in ["experiments", "oof", "submissions"]:
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)

with open(DATA_DIR / "final_posts.json", encoding="utf-8") as fh:
    raw_posts = json.load(fh)
print("final_posts.json is a", type(raw_posts).__name__, "of length", len(raw_posts))
print("keys of the first record:", list(raw_posts[0].keys()))

posts = pd.DataFrame(raw_posts)
identification = pd.read_csv(DATA_DIR / "data_identification.csv")
emotion = pd.read_csv(DATA_DIR / "emotion.csv")
sample_submission = pd.read_csv(DATA_DIR / "sampleSubmission.csv")

for name, frame in [("posts", posts), ("identification", identification),
                    ("emotion", emotion), ("sample_submission", sample_submission)]:
    print(f"{name:18s} shape={str(frame.shape):12s} columns={list(frame.columns)}")

print("\nposts.post_id is unique:", posts["post_id"].is_unique)
print("missing values in posts:", posts.isna().sum().to_dict())
print("train/test split recorded in data_identification.csv:",
      identification["set"].value_counts().to_dict())


final_posts.json is a list of length 4800
keys of the first record: ['post_id', 'text']
posts              shape=(4800, 2)    columns=['post_id', 'text']
identification     shape=(4800, 2)    columns=['post_id', 'set']
emotion            shape=(3360, 2)    columns=['post_id', 'label']
sample_submission  shape=(1440, 2)    columns=['post_id', 'label']

posts.post_id is unique: True
missing values in posts: {'post_id': 0, 'text': 0}
train/test split recorded in data_identification.csv: {'train': 3360, 'test': 1440}


In [2]:
# T0 step 2 - merge the three files on post_id and assert the structure we were promised.
# A silent merge failure here would poison everything downstream, so each expectation
# is checked with an assert rather than just printed and eyeballed.
merged = posts.merge(identification, on="post_id", how="outer", indicator=True)
print("merge coverage posts vs identification:", merged["_merge"].value_counts().to_dict())
assert (merged["_merge"] == "both").all(), "some post_id appears in only one of the two files"

merged = merged.drop(columns="_merge").merge(emotion, on="post_id", how="left")
train = merged[merged["set"] == "train"].reset_index(drop=True)
test = merged[merged["set"] == "test"].reset_index(drop=True)

print(f"\ntrain rows {len(train)} + test rows {len(test)} = {len(train) + len(test)}")
assert len(train) + len(test) == 4800, "train + test should cover all 4,800 posts"
assert train["label"].notna().all(), "every training row must carry a label"
assert test["label"].isna().all(), "no test row may carry a label"
assert len(test) == len(sample_submission), "test rows must match sampleSubmission rows"
assert set(test["post_id"]) == set(sample_submission["post_id"]), "test ids must match the sample file"
print("test ids match sampleSubmission as a set:", True)
print("test ids are in the same ORDER as sampleSubmission:",
      list(test["post_id"]) == list(sample_submission["post_id"]))
print("-> the submission must therefore be re-ordered to the sample file, not written in test order")

LABELS = sorted(train["label"].unique())
print("\nthe six labels, exactly as spelled in emotion.csv:", LABELS)


merge coverage posts vs identification: {'both': 4800, 'left_only': 0, 'right_only': 0}

train rows 3360 + test rows 1440 = 4800
test ids match sampleSubmission as a set: True
test ids are in the same ORDER as sampleSubmission: False
-> the submission must therefore be re-ordered to the sample file, not written in test order

the six labels, exactly as spelled in emotion.csv: ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']


In [3]:
# T0 step 3 - describe the data, and turn the class distribution into the three numbers
# that the probe submission will be read against.
# Mean F1 is not defined anywhere on the competition pages (macro / micro / weighted all
# fit the words), so one throw-away submission that predicts a single class everywhere
# will tell us which one it is. If the largest class covers a share p of the scored subset,
# that submission gets precision = p, recall = 1, so its F1 for that class is 2p/(1+p),
# and the three candidate metrics give three very different totals.
counts = train["label"].value_counts()
distribution = pd.DataFrame({
    "n": counts,
    "share_%": (counts / len(train) * 100).round(2),
})
print("=== class distribution in the training set ===")
print(distribution.to_string())
print("\nsmallest class:", counts.idxmin(), counts.min(),
      "| largest class:", counts.idxmax(), counts.max(),
      f"| imbalance ratio {counts.max() / counts.min():.2f}x")

MAJORITY_LABEL = counts.idxmax()
p = counts.max() / len(train)
f1_of_that_class = 2 * p / (1 + p)
probe_expectation = {
    "macro": f1_of_that_class / 6,
    "micro": p,
    "weighted": p * f1_of_that_class,
}
print(f"\n=== what a constant '{MAJORITY_LABEL}' submission should score (p = {p:.4f}) ===")
for metric, value in probe_expectation.items():
    print(f"  if Mean F1 means {metric:9s} -> {value:.4f}")
print(f"  weighted / macro = {probe_expectation['weighted'] / probe_expectation['macro']:.2f}x,"
      " far enough apart to decide from a single submission")
print("  caveat: p is estimated from the training split; the public subset is only 30% of the"
      " test set and its true class shares are unknown.")

train_lengths = pd.DataFrame({
    "chars": train["text"].str.len(),
    "words": train["text"].str.split().str.len(),
})
print("\n=== text length, training set ===")
print(train_lengths.describe().round(2).to_string())
print("\n=== words per post, by class ===")
print(train.assign(words=train_lengths["words"])
          .groupby("label")["words"].agg(["mean", "median", "max"]).round(2).to_string())
print("\n-> the length distribution is heavily right-skewed (median 13 words, maximum above 500),"
      " which is why TF-IDF is used with sublinear term frequencies below.")


=== class distribution in the training set ===
            n  share_%
label                 
joy       875    26.04
anger     674    20.06
sadness   628    18.69
disgust   454    13.51
surprise  441    13.12
fear      288     8.57

smallest class: fear 288 | largest class: joy 875 | imbalance ratio 3.04x

=== what a constant 'joy' submission should score (p = 0.2604) ===
  if Mean F1 means macro     -> 0.0689
  if Mean F1 means micro     -> 0.2604
  if Mean F1 means weighted  -> 0.1076
  weighted / macro = 1.56x, far enough apart to decide from a single submission
  caveat: p is estimated from the training split; the public subset is only 30% of the test set and its true class shares are unknown.

=== text length, training set ===
         chars    words
count  3360.00  3360.00
mean     87.78    16.51
std     121.88    24.96
min       5.00     1.00
25%      42.00     8.00
50%      68.00    13.00
75%     101.25    19.00
max    2545.00   508.00

=== words per post, by class ===
         

In [4]:
# T0 step 4 - build the ONE fixed cross-validation split that every later experiment reuses.
# The four steps below must happen in this order:
#   1. normalise the text (lowercase, collapse whitespace)
#   2. count duplicate texts
#   3. if duplicates exist, group them so that copies of one text cannot be split across folds
#   4. only then write folds.csv
# Skipping step 3 would let the same sentence sit in both the training and the validation
# fold, which inflates every out-of-fold score and makes all later comparisons worthless.
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold

def normalise(text):
    """Lowercase and collapse runs of whitespace - used only for duplicate detection."""
    return re.sub(r"\s+", " ", str(text).lower()).strip()

train["text_normalised"] = train["text"].map(normalise)
test["text_normalised"] = test["text"].map(normalise)

train_dupes = train["text_normalised"].value_counts()
train_dupes = train_dupes[train_dupes > 1]
test_dupes = test["text_normalised"].value_counts()
test_dupes = test_dupes[test_dupes > 1]
label_disagreement = train.groupby("text_normalised")["label"].nunique()

print("=== duplicate texts after normalisation ===")
print(f"train: {len(train_dupes)} distinct texts appear more than once, covering"
      f" {int(train_dupes.sum())} rows (largest group {int(train_dupes.max())} rows)")
print(f"test : {len(test_dupes)} distinct texts appear more than once, covering"
      f" {int(test_dupes.sum())} rows")
print(f"identical training texts carrying DIFFERENT labels: {(label_disagreement > 1).sum()}")
print("-> those contradictory pairs put a ceiling on any achievable score, and they are the"
      " reason the fold split has to be grouped.")

overlap = set(train["text_normalised"]) & set(test["text_normalised"])
print(f"\ntexts that occur in BOTH train and test: {len(overlap)} distinct"
      f" ({train['text_normalised'].isin(overlap).sum()} train rows,"
      f" {test['text_normalised'].isin(overlap).sum()} test rows)")
print("-> recorded as a limitation in the report: part of the test set is memorisable.")

train["group"] = train["text_normalised"].map(
    lambda s: hashlib.md5(s.encode("utf-8")).hexdigest()
)
if len(train_dupes) > 0:
    splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    split_iter = splitter.split(train, train["label"], groups=train["group"])
    print("\nduplicates found -> using StratifiedGroupKFold with the text hash as the group")
else:
    splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    split_iter = splitter.split(train, train["label"])
    print("\nno duplicates -> plain StratifiedKFold is enough")

train["fold"] = -1
for fold_id, (_, validation_index) in enumerate(split_iter):
    train.loc[validation_index, "fold"] = fold_id
assert (train["fold"] >= 0).all(), "every training row must land in exactly one fold"

print("\nrows per fold:", train["fold"].value_counts().sort_index().to_dict())
print("groups that leaked across more than one fold:",
      int((train.groupby("group")["fold"].nunique() > 1).sum()), "(must be 0)")
print("\nclass share per fold, in % (stratification check):")
print((pd.crosstab(train["fold"], train["label"], normalize="index") * 100).round(2).to_string())

train[["post_id", "label", "fold"]].to_csv(OUT_DIR / "folds.csv", index=False)
print(f"\nwritten: {OUT_DIR / 'folds.csv'} - this file is now frozen for the whole competition")


=== duplicate texts after normalisation ===
train: 50 distinct texts appear more than once, covering 100 rows (largest group 2 rows)
test : 8 distinct texts appear more than once, covering 16 rows
identical training texts carrying DIFFERENT labels: 16
-> those contradictory pairs put a ceiling on any achievable score, and they are the reason the fold split has to be grouped.

texts that occur in BOTH train and test: 59 distinct (61 train rows, 59 test rows)
-> recorded as a limitation in the report: part of the test set is memorisable.

duplicates found -> using StratifiedGroupKFold with the text hash as the group

rows per fold: {0: 673, 1: 672, 2: 672, 3: 672, 4: 671}
groups that leaked across more than one fold: 0 (must be 0)

class share per fold, in % (stratification check):
label  anger  disgust  fear    joy  sadness  surprise
fold                                                 
0      20.06    13.52  8.62  26.00    18.72     13.08
1      20.09    13.54  8.48  26.04    18.75    

## 2. Feature Engineering Steps

In [5]:
### Add the code related to the feature engineering steps in cells inside this section

# Feature family 1 and 2 - two TF-IDF views of the same text, horizontally stacked.
#   word level  (1,2)-grams: catches vocabulary and short phrases such as "not happy"
#   char level  char_wb (2,5)-grams: catches lengthened words ("angryyy"), misspellings and
#               emoji, which the word view throws away as unseen tokens
# Both vectorisers are fitted INSIDE each fold on the training part only. Fitting them once
# on all 3,360 rows would let validation-fold vocabulary and document frequencies leak into
# the features and would quietly inflate every score.
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer

WORD_TFIDF = dict(ngram_range=(1, 2), min_df=2, sublinear_tf=True, strip_accents="unicode")
CHAR_TFIDF = dict(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)

# A one-off look at how wide each view is when fitted on four of the five folds.
demo_train_text = train.loc[train["fold"] != 0, "text"].values
for view_name, settings in [("word", WORD_TFIDF), ("char_wb", CHAR_TFIDF)]:
    matrix = TfidfVectorizer(**settings).fit_transform(demo_train_text)
    density = matrix.nnz / (matrix.shape[0] * matrix.shape[1]) * 100
    print(f"{view_name:8s} view: {matrix.shape[1]:6d} features, {density:.3f}% of entries non-zero")
print("\n-> the character view is roughly four times wider than the word view and is the"
      " cheapest single improvement available at this stage.")


word     view:   6107 features, 0.263% of entries non-zero
char_wb  view:  24566 features, 0.755% of entries non-zero

-> the character view is roughly four times wider than the word view and is the cheapest single improvement available at this stage.


In [6]:
# Feature family 3 - MiniLM sentence embeddings.
# This is the same all-MiniLM-L6-v2 model used in Master section 5, so no new download is
# needed. Unlike TF-IDF it carries meaning learned from a large external corpus, so two posts
# can be close even with no shared words; that is exactly what the sparse views cannot do.
# The encoder is UNSUPERVISED with respect to our labels, so encoding all 4,800 texts once -
# outside the fold loop - leaks nothing, and it keeps the cross-validation cheap.
from sentence_transformers import SentenceTransformer

EMBEDDING_CACHE = OUT_DIR / "minilm_embeddings.npz"
if EMBEDDING_CACHE.exists():
    cached = np.load(EMBEDDING_CACHE)
    train_embeddings, test_embeddings = cached["train"], cached["test"]
    print("loaded cached embeddings from", EMBEDDING_CACHE)
else:
    encoder = SentenceTransformer("all-MiniLM-L6-v2")
    started = time.time()
    train_embeddings = encoder.encode(list(train["text"]), batch_size=64,
                                      normalize_embeddings=True, show_progress_bar=False)
    test_embeddings = encoder.encode(list(test["text"]), batch_size=64,
                                     normalize_embeddings=True, show_progress_bar=False)
    np.savez(EMBEDDING_CACHE, train=train_embeddings, test=test_embeddings)
    print(f"encoded {len(train) + len(test)} texts in {time.time() - started:.0f}s"
          f" and cached them to {EMBEDDING_CACHE}")

print("train embeddings", train_embeddings.shape, "| test embeddings", test_embeddings.shape)
print("rows are L2-normalised:", np.allclose(np.linalg.norm(train_embeddings, axis=1), 1.0))

EMBEDDING_SCALE = 1.0   # embeddings sit beside TF-IDF columns; this keeps the two blocks comparable

def build_features(train_text, apply_text, train_embedding, apply_embedding, views):
    """Fit the requested views on train_text only, then apply them to apply_text."""
    fitted_train, fitted_apply = [], []
    if "word" in views:
        vectoriser = TfidfVectorizer(**WORD_TFIDF)
        fitted_train.append(vectoriser.fit_transform(train_text))
        fitted_apply.append(vectoriser.transform(apply_text))
    if "char" in views:
        vectoriser = TfidfVectorizer(**CHAR_TFIDF)
        fitted_train.append(vectoriser.fit_transform(train_text))
        fitted_apply.append(vectoriser.transform(apply_text))
    if "emb" in views:
        fitted_train.append(csr_matrix(train_embedding * EMBEDDING_SCALE))
        fitted_apply.append(csr_matrix(apply_embedding * EMBEDDING_SCALE))
    if len(fitted_train) == 1:
        return fitted_train[0], fitted_apply[0]
    return hstack(fitted_train).tocsr(), hstack(fitted_apply).tocsr()

print("\nbuild_features() is the single place where features are made, so the"
      " cross-validation below and the final full-data fit cannot drift apart.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

encoded 4800 texts in 4s and cached them to ..\_workbench\kaggle\minilm_embeddings.npz
train embeddings (3360, 384) | test embeddings (1440, 384)
rows are L2-normalised: True

build_features() is the single place where features are made, so the cross-validation below and the final full-data fit cannot drift apart.


## 3. Model Implementation Steps

In [7]:
### Add the code related to the model implementation steps in cells inside this section

# Model search - out-of-fold macro F1 over the frozen 5-fold split from section 1.
# Every configuration is scored on exactly the same folds, so the numbers are comparable and
# the out-of-fold predictions could later be reused for an ensemble.
# Three things are varied, one at a time: which feature views are used, whether the classes
# are reweighted, and the regularisation strength C.
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore", category=UserWarning)

y_train = train["label"].values
fold_ids = train["fold"].values
texts = train["text"].values

def evaluate(views, class_weight, C):
    """Return out-of-fold predictions and the three candidate F1 averages."""
    out_of_fold = np.empty(len(train), dtype=object)
    per_fold = []
    started = time.time()
    for fold_id in range(N_SPLITS):
        is_train = fold_ids != fold_id
        is_validation = fold_ids == fold_id
        X_train, X_validation = build_features(
            texts[is_train], texts[is_validation],
            train_embeddings[is_train], train_embeddings[is_validation], views,
        )
        model = LinearSVC(C=C, class_weight=class_weight,
                          random_state=RANDOM_STATE, max_iter=10000)
        model.fit(X_train, y_train[is_train])
        predictions = model.predict(X_validation)
        out_of_fold[is_validation] = predictions
        per_fold.append(f1_score(y_train[is_validation], predictions, average="macro"))
    return out_of_fold, {
        "views": "+".join(views),
        "class_weight": str(class_weight),
        "C": C,
        "oof_macro_f1": f1_score(y_train, out_of_fold, average="macro"),
        "oof_micro_f1": f1_score(y_train, out_of_fold, average="micro"),
        "oof_weighted_f1": f1_score(y_train, out_of_fold, average="weighted"),
        "fold_mean": float(np.mean(per_fold)),
        "fold_std": float(np.std(per_fold)),
        "seconds": round(time.time() - started, 1),
    }

CANDIDATES = [
    (["word"],                "balanced", 0.10),   # single sparse view, for reference
    (["char"],                "balanced", 0.10),   # characters alone already beat words
    (["emb"],                 "balanced", 1.00),   # dense view alone
    (["word", "char"],        None,       0.10),   # does reweighting help?
    (["word", "char"],        "balanced", 0.10),
    (["word", "char"],        "balanced", 0.20),   # does less regularisation help?
    (["word", "char", "emb"], "balanced", 0.05),
    (["word", "char", "emb"], "balanced", 0.10),
]

records, oof_store = [], {}
for views, class_weight, C in CANDIDATES:
    predictions, record = evaluate(views, class_weight, C)
    key = f"{record['views']}|cw={record['class_weight']}|C={C}"
    oof_store[key] = predictions
    records.append(record)
    print(f"{key:40s} macro={record['oof_macro_f1']:.4f} (folds "
          f"{record['fold_mean']:.4f} +/- {record['fold_std']:.4f})  "
          f"micro={record['oof_micro_f1']:.4f}  weighted={record['oof_weighted_f1']:.4f}  "
          f"{record['seconds']}s", flush=True)

results = pd.DataFrame(records).sort_values("oof_macro_f1", ascending=False)
results.to_csv(OUT_DIR / "experiments" / "t1_grid.csv", index=False)
print("\n=== ranked by out-of-fold macro F1 ===")
print(results[["views", "class_weight", "C", "oof_macro_f1", "fold_std",
               "oof_micro_f1", "oof_weighted_f1"]].to_string(index=False))

best = results.iloc[0]
print(f"\nbest configuration: views={best['views']}, class_weight={best['class_weight']},"
      f" C={best['C']} -> out-of-fold macro F1 {best['oof_macro_f1']:.4f}"
      f" (fold standard deviation {best['fold_std']:.4f})")
print("the top configurations sit within one fold standard deviation of each other, so the"
      " ranking inside that group is noise; what is NOT noise is the gap opened by adding"
      " the embedding view.")


word|cw=balanced|C=0.1                   macro=0.4730 (folds 0.4728 +/- 0.0134)  micro=0.4940  weighted=0.4915  0.3s
char|cw=balanced|C=0.1                   macro=0.5045 (folds 0.5041 +/- 0.0116)  micro=0.5229  weighted=0.5201  1.7s
emb|cw=balanced|C=1.0                    macro=0.4863 (folds 0.4856 +/- 0.0091)  micro=0.5071  weighted=0.5075  1.7s
word+char|cw=None|C=0.1                  macro=0.5096 (folds 0.5086 +/- 0.0188)  micro=0.5357  weighted=0.5248  2.1s
word+char|cw=balanced|C=0.1              macro=0.5129 (folds 0.5122 +/- 0.0133)  micro=0.5280  weighted=0.5264  1.9s
word+char|cw=balanced|C=0.2              macro=0.5146 (folds 0.5138 +/- 0.0159)  micro=0.5307  weighted=0.5292  2.1s
word+char+emb|cw=balanced|C=0.05         macro=0.5708 (folds 0.5705 +/- 0.0167)  micro=0.5878  weighted=0.5851  2.6s
word+char+emb|cw=balanced|C=0.1          macro=0.5657 (folds 0.5654 +/- 0.0154)  micro=0.5839  weighted=0.5819  2.7s

=== ranked by out-of-fold macro F1 ===
        views class_weig

In [8]:
# Refit the chosen configuration on all 3,360 labelled rows and predict the test set.
# The vectorisers are fitted here on the full training text - that is correct, because the
# test labels are never touched; only the fold-level fits had to stay inside the fold.
BEST_VIEWS = ["word", "char", "emb"]
BEST_CLASS_WEIGHT = "balanced"
BEST_C = 0.05

X_full, X_test = build_features(train["text"].values, test["text"].values,
                                train_embeddings, test_embeddings, BEST_VIEWS)
print("full training matrix", X_full.shape, "| test matrix", X_test.shape)

final_model = LinearSVC(C=BEST_C, class_weight=BEST_CLASS_WEIGHT,
                        random_state=RANDOM_STATE, max_iter=10000)
final_model.fit(X_full, y_train)
test_predictions = final_model.predict(X_test)

print("\npredicted class distribution on the test set, next to the training distribution:")
comparison = pd.DataFrame({
    "test_predicted_%": (pd.Series(test_predictions).value_counts(normalize=True) * 100).round(2),
    "train_actual_%": (train["label"].value_counts(normalize=True) * 100).round(2),
})
print(comparison.to_string())
print("\n-> class_weight='balanced' pushes the rare classes up, so the predicted shares are"
      " flatter than the training shares. That is deliberate: macro F1 rewards the small"
      " classes as much as the large ones.")

# Keep the out-of-fold predictions of the chosen configuration for the report and for any
# later ensemble work.
best_key = f"word+char+emb|cw={BEST_CLASS_WEIGHT}|C={BEST_C}"
oof_frame = pd.DataFrame({
    "post_id": train["post_id"], "true_label": y_train,
    "oof_prediction": oof_store[best_key], "fold": fold_ids,
})
oof_frame.to_csv(OUT_DIR / "oof" / "tfidf_char_minilm_linearsvc.csv", index=False)

print("\n=== per-class out-of-fold performance of the chosen configuration ===")
print(classification_report(y_train, oof_store[best_key], digits=3))
print("confusion matrix (rows = true label, columns = predicted):")
print(pd.DataFrame(confusion_matrix(y_train, oof_store[best_key], labels=LABELS),
                   index=LABELS, columns=LABELS).to_string())


full training matrix (3360, 35420) | test matrix (1440, 35420)

predicted class distribution on the test set, next to the training distribution:
          test_predicted_%  train_actual_%
anger                22.01           20.06
disgust              10.83           13.51
fear                  8.61            8.57
joy                  28.68           26.04
sadness              16.88           18.69
surprise             12.99           13.12

-> class_weight='balanced' pushes the rare classes up, so the predicted shares are flatter than the training shares. That is deliberate: macro F1 rewards the small classes as much as the large ones.

=== per-class out-of-fold performance of the chosen configuration ===
              precision    recall  f1-score   support

       anger      0.545     0.580     0.562       674
     disgust      0.507     0.421     0.460       454
        fear      0.609     0.580     0.594       288
         joy      0.668     0.727     0.696       875
     sadness

In [9]:
# Write the submission files, and refuse to write anything that fails the five checks.
# Two header spellings are produced on purpose: the Overview page and the Kaggle API both
# say the identifier column is "id", while the downloaded sampleSubmission.csv header says
# "post_id". The probe submission is uploaded first and settles which one Kaggle accepts.
SUBMISSION_DIR = OUT_DIR / "submissions"

def check_and_write(post_ids, labels, stem):
    """Run the five pre-submission checks, then write the id and post_id header variants."""
    frame = pd.DataFrame({"post_id": post_ids, "label": labels})
    # the scored file must follow sampleSubmission's row order, so join onto it
    frame = sample_submission[["post_id"]].merge(frame, on="post_id", how="left")

    problems = []
    if list(frame.columns) != ["post_id", "label"]:
        problems.append(f"1. expected exactly two columns, got {list(frame.columns)}")
    if len(frame) != len(sample_submission):
        problems.append(f"2. expected {len(sample_submission)} rows, got {len(frame)}")
    unexpected = set(frame["label"].dropna()) - set(LABELS)
    if unexpected:
        problems.append(f"3. labels outside the six classes: {unexpected}")
    if frame.isna().any().any():
        problems.append(f"4. missing values: {frame.isna().sum().to_dict()}")
    if not set(frame["label"].dropna()) <= set(emotion["label"].unique()):
        problems.append("5. label strings do not match emotion.csv exactly")
    if problems:
        raise ValueError("submission rejected:\n  " + "\n  ".join(problems))

    written = []
    for id_column in ["id", "post_id"]:
        out = frame.rename(columns={"post_id": id_column})
        path = SUBMISSION_DIR / f"{stem}__{id_column}.csv"
        out.to_csv(path, index=False)
        written.append(path)
    print(f"all five checks passed for '{stem}' "
          f"({len(frame)} rows, {frame['label'].nunique()} distinct labels)")
    for path in written:
        print("   written:", path)
    return frame

# Submission A - the probe. Predicts the majority class everywhere, purely to read off which
# average "Mean F1" refers to. It must never be chosen as a final submission.
probe = check_and_write(test["post_id"].values,
                        np.full(len(test), MAJORITY_LABEL), "probe_constant")
print(f"\nPROBE: every row predicted '{MAJORITY_LABEL}'. Expected score if the metric is"
      f" macro {probe_expectation['macro']:.4f} / micro {probe_expectation['micro']:.4f}"
      f" / weighted {probe_expectation['weighted']:.4f}")

# Submission B - the real first entry.
submission = check_and_write(test["post_id"].values, test_predictions, "t1_tfidf_char_minilm")
print(f"\nREAL ENTRY: out-of-fold macro F1 {best['oof_macro_f1']:.4f};"
      " the first three rows of the file are")
print(submission.head(3).to_string(index=False))

print("\nUpload order: the probe first (it settles both the metric and the header), then the"
      " real entry. Both go in the submission ledger before anything is uploaded.")


all five checks passed for 'probe_constant' (1440 rows, 1 distinct labels)
   written: ..\_workbench\kaggle\submissions\probe_constant__id.csv
   written: ..\_workbench\kaggle\submissions\probe_constant__post_id.csv

PROBE: every row predicted 'joy'. Expected score if the metric is macro 0.0689 / micro 0.2604 / weighted 0.1076
all five checks passed for 't1_tfidf_char_minilm' (1440 rows, 6 distinct labels)
   written: ..\_workbench\kaggle\submissions\t1_tfidf_char_minilm__id.csv
   written: ..\_workbench\kaggle\submissions\t1_tfidf_char_minilm__post_id.csv

REAL ENTRY: out-of-fold macro F1 0.5708; the first three rows of the file are
post_id   label
xyzx9jp   anger
l3vzzxt sadness
g680iug sadness

Upload order: the probe first (it settles both the metric and the header), then the real entry. Both go in the submission ledger before anything is uploaded.
